# Notebook 06 — Hypothesis Testing

**Project:** Autobahn Speed-Safety Analysis  
**Context:** The previous analyses (notebooks 03–05) established a higher fatality severity index on unlimited Autobahn sections compared to speed-limited sections. This notebook applies three literature-grounded statistical methods to test causal hypotheses.

## Hypotheses

| ID | Name | Method | Reference |
|----|------|--------|-----------|
| H3 | Diverging safety trends | Poisson regression on annual rates | Johansson (1996) |
| H4 | Power Model back-calculation | Speed-based ratio prediction | Elvik (2013) |
| H5 | Crash type distribution | Safe System threshold analysis | Doecke (2018) |

## Data

- **Source:** `data/processed/unfallatlas_classified.parquet` — German accident microdata 2016–2024
- **Key columns:** `year`, `severity` (fatal/serious_injury/slight_injury), `on_unlimited`, `on_limited_mw`, `UTYP1` (collision situation)
- **Network constants:** Unlimited = 16,441 km; Limited = 10,618 km (OSM, pre-computed)

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.genmod.families as fam
from scipy import stats

sns.set_theme(style='whitegrid')

ROOT      = Path('..').resolve()
DATA_PROC = ROOT / 'data' / 'processed'
FIG_DIR   = ROOT / 'output' / 'figures'

# Network length constants (OSM, pre-computed)
KM_UNLIMITED = 16_441
KM_LIMITED   = 10_618

df = pd.read_parquet(DATA_PROC / 'unfallatlas_classified.parquet')
print(f'Loaded {len(df):,} accident records, years: {sorted(df["year"].unique())}')
df.head(3)

---
## H3: Diverging Safety Trends (Johansson 1996)

### Hypothesis
Speed-limited Autobahn sections show a statistically significant downward trend in fatality rates over 2016–2024, while unlimited sections do not improve (or improve more slowly). A diverging trend would suggest that speed limits are actively improving safety on the limited sections.

### Method
Following Johansson (1996), we fit Poisson regression models on annual fatal accident counts:

$$\log(\mu_t) = \alpha + \beta_{\text{year}} \cdot t + \log(N_t)$$

where $N_t$ is the total number of accidents in year $t$ (used as the exposure offset). This models the *fatality rate* (fatal/total), not the raw count.

We fit:
1. Separate models for each group → $\beta_{\text{year}}$, p-value, % change per year
2. A combined interaction model: `fatal ~ year + group + year:group` → tests if trends **diverge significantly**

In [ ]:
def build_annual(df_sub: pd.DataFrame) -> pd.DataFrame:
    by_year = (
        df_sub.groupby('year')
        .agg(total=('severity', 'count'),
             fatal=('severity', lambda x: (x == 'fatal').sum()))
        .reset_index()
    )
    by_year['fatal_rate'] = by_year['fatal'] / by_year['total']
    by_year['log_offset'] = np.log(by_year['total'])
    by_year['year_c']     = by_year['year'] - by_year['year'].mean()
    return by_year

df_unl = build_annual(df[df['on_unlimited']])
df_lim = build_annual(df[df['on_limited_mw']])

# Separate Poisson regressions
results = {}
for label, df_grp in [('Unlimited', df_unl), ('Limited', df_lim)]:
    model = smf.glm(
        'fatal ~ year_c',
        data=df_grp,
        family=fam.Poisson(),
        offset=df_grp['log_offset'],
    ).fit(disp=False)
    beta    = model.params['year_c']
    pval    = model.pvalues['year_c']
    pct_yr  = (np.exp(beta) - 1) * 100
    results[label] = dict(model=model, beta=beta, pval=pval, pct_yr=pct_yr, df_grp=df_grp)
    print(f'{label}: β_year = {beta:.4f}, p = {pval:.4f}, %change/yr = {pct_yr:+.2f}%')

# Combined interaction model
df_unl_c = df_unl.copy(); df_unl_c['group'] = 1
df_lim_c = df_lim.copy(); df_lim_c['group'] = 0
df_comb  = pd.concat([df_unl_c, df_lim_c], ignore_index=True)
model_comb = smf.glm(
    'fatal ~ year_c + group + year_c:group',
    data=df_comb,
    family=fam.Poisson(),
    offset=df_comb['log_offset'],
).fit(disp=False)
int_beta = model_comb.params['year_c:group']
int_p    = model_comb.pvalues['year_c:group']
print(f'\nInteraction (year × group): β = {int_beta:.4f}, p = {int_p:.4f}')
print(f"→ {'SIGNIFICANT' if int_p < 0.05 else 'Not significant'} divergence (α=0.05)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
colors  = {'Unlimited': '#e63946', 'Limited': '#457b9d'}
markers = {'Unlimited': 'o',       'Limited': 's'}

for label, res in results.items():
    df_g   = res['df_grp']
    fitted = res['model'].predict(df_g) / df_g['total']
    ax.scatter(df_g['year'], df_g['fatal_rate'] * 1000,
               color=colors[label], marker=markers[label], s=60, zorder=5,
               label=f'{label} (observed)')
    ax.plot(df_g['year'], fitted * 1000, '--', color=colors[label], linewidth=2,
            label=f"{label} fitted: {res['pct_yr']:+.1f}%/yr (p={res['pval']:.3f})")

ax.set_title(
    'H3: Annual fatality rate with Poisson trend lines (Johansson 1996)\n'
    f"Unlimited: {results['Unlimited']['pct_yr']:+.1f}%/yr (p={results['Unlimited']['pval']:.3f})  |  "
    f"Limited: {results['Limited']['pct_yr']:+.1f}%/yr (p={results['Limited']['pval']:.3f})\n"
    f'Interaction β={int_beta:.4f} (p={int_p:.3f}) — difference in trends'
)
ax.set_xlabel('Year'); ax.set_ylabel('Fatalities per 1000 accidents')
ax.legend(fontsize=9); sns.despine(); plt.tight_layout()
plt.savefig(FIG_DIR / 'fig10_trend_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### H3 Interpretation

| Group | β_year | p-value | % change / year |
|-------|--------|---------|----------------|
| Unlimited Autobahn | −0.0269 | 0.008 | −2.7% |
| Speed-limited Autobahn | −0.0475 | 0.001 | −4.6% |

- **Both** groups show statistically significant downward trends in fatality rates.
- Limited sections improve **faster** (−4.6%/yr vs −2.7%/yr), consistent with H3.
- However, the **interaction term** (β = 0.021, p = 0.24) is **not significant** — the difference in trends cannot be distinguished from noise at α = 0.05 with only 9 years of data.
- **Conclusion:** H3 is directionally supported but not statistically confirmed. A longer time series (post-2024) would be needed.

---
## H4: Power Model Back-Calculation (Elvik 2013)

### Hypothesis
The observed fatality ratio (unlimited vs limited) is consistent with Power Model predictions, confirming that speed is the primary causal mechanism for the observed gap.

### Method
The Power Model (Nilsson 2004, Elvik 2013) predicts accident counts change as a power function of speed:

$$\frac{A_2}{A_1} = \left(\frac{V_2}{V_1}\right)^n$$

**Speed constants (BASt 2019 — Verkehr auf Bundesautobahnen annual report):**
- Unlimited sections: **135 km/h** (mean speed, passenger cars, sections without limit)
- Limited sections: **118 km/h** (mean speed, sections with 120–130 km/h limit)

**Elvik (2013) motorway-specific exponents:**
- Fatal accidents: n = 4.1 (95% CI: 2.9–5.3)
- Serious injury: n = 2.6
- All injury: n = 1.6

We then compare these *predicted* ratios to the *observed* per-km accident rate ratios from the Unfallatlas.

In [ ]:
# Speed constants (BASt 2019)
V_UNLIMITED = 135.0   # km/h — unlimited sections
V_LIMITED   = 118.0   # km/h — limited sections
SPEED_RATIO = V_UNLIMITED / V_LIMITED

# Elvik (2013) exponents
EXPONENTS = {
    'Fatal accidents':          {'exp': 4.1, 'lo': 2.9, 'hi': 5.3},
    'Serious injury accidents':  {'exp': 2.6, 'lo': 2.6, 'hi': 2.6},
    'All injury accidents':      {'exp': 1.6, 'lo': 1.6, 'hi': 1.6},
}

# Observed per-km rates
mw_unl = df[df['on_unlimited']]
mw_lim = df[df['on_limited_mw']]

OBSERVED = {
    'Fatal accidents': (mw_unl['severity'] == 'fatal').sum() / KM_UNLIMITED /
                       ((mw_lim['severity'] == 'fatal').sum() / KM_LIMITED),
    'Serious injury accidents': (mw_unl['severity'] == 'serious_injury').sum() / KM_UNLIMITED /
                                ((mw_lim['severity'] == 'serious_injury').sum() / KM_LIMITED),
    'All injury accidents':     len(mw_unl) / KM_UNLIMITED / (len(mw_lim) / KM_LIMITED),
}

rows = []
print(f'{"Severity":<30} {"Predicted":>10} {"CI low":>8} {"CI high":>8} {"Observed":>10} {"Interpretation"}')
print('-' * 90)
for label, ex in EXPONENTS.items():
    pred_mid = SPEED_RATIO ** ex['exp']
    pred_lo  = SPEED_RATIO ** ex['lo']
    pred_hi  = SPEED_RATIO ** ex['hi']
    obs_val  = OBSERVED[label]
    rows.append(dict(severity=label, predicted=pred_mid, pred_lo=pred_lo,
                     pred_hi=pred_hi, observed=obs_val))
    interp = ('< predicted → confounding advantages' if obs_val < pred_lo
               else ('≈ predicted → speed explains gap' if obs_val <= pred_hi
                     else '> predicted → additional risks'))
    print(f'{label:<30} {pred_mid:>10.3f} {pred_lo:>8.3f} {pred_hi:>8.3f} {obs_val:>10.3f}  {interp}')

df_h4 = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x, w = np.arange(len(df_h4)), 0.32

bars_pred = ax.bar(x - w/2, df_h4['predicted'], w, color='#457b9d', label='Power Model prediction', alpha=0.85)
bars_obs  = ax.bar(x + w/2, df_h4['observed'],  w, color='#e63946', label='Observed (per km)',     alpha=0.85)

err_lo = df_h4['predicted'] - df_h4['pred_lo']
err_hi = df_h4['pred_hi']   - df_h4['predicted']
ax.errorbar(x - w/2, df_h4['predicted'], yerr=[err_lo, err_hi],
            fmt='none', color='black', capsize=5, linewidth=1.5, label='95% CI (exponent)')

ax.bar_label(bars_pred, fmt='%.2f', padding=4, fontsize=9, fontweight='bold')
ax.bar_label(bars_obs,  fmt='%.2f', padding=4, fontsize=9, fontweight='bold')
ax.axhline(1.0, color='grey', linestyle=':', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(df_h4['severity'], fontsize=10)
ax.set_ylabel('Rate ratio (unlimited / limited, per km)')
ax.set_title(f'H4: Power Model vs Observed (speed: {V_UNLIMITED} vs {V_LIMITED} km/h — BASt 2019)')
ax.legend(); sns.despine(); plt.tight_layout()
plt.show()

### H4 Interpretation

| Severity | Predicted ratio | 95% CI | Observed ratio | Result |
|----------|----------------|--------|----------------|--------|
| Fatal | 1.74 | 1.48–2.04 | 1.27 | **< predicted** |
| Serious injury | 1.42 | — | 1.02 | **< predicted** |
| All injury | 1.24 | — | 0.89 | **< predicted** |

- All observed ratios fall **below** the Power Model predictions — even below the 95% CI lower bound for fatals.
- This suggests unlimited sections benefit from **confounding advantages**: lower traffic density, better road geometry, wider lanes, absence of junctions and pedestrian crossings.
- **Conclusion:** H4 is **not confirmed** — speed alone does not explain the full observed gap. The Power Model over-predicts the difference, meaning unlimited sections have structural advantages that partially offset the speed penalty. A proper causal estimate would need to control for these confounders.

---
## H5: Crash Type Distribution — Safe System Threshold (Doecke 2018)

### Hypothesis
Unlimited sections have a **higher proportion** of high-energy crash types (head-on: UTYP1=3, run-off road: UTYP1=7) relative to lower-energy types. These crash types are unsafe at any speed above ~50–80 km/h (Doecke 2018), so their relative frequency matters.

### Method
1. For motorway accidents, compute the distribution of `UTYP1` collision types.
2. Group into: **High-energy** (UTYP1=3,7), **Medium-energy** (UTYP1=2,6), **Other**.
3. Chi-squared test for independence of distributions.
4. Compute case fatality rate (CFR = fatal/total) by crash type × speed regime.

**UTYP1 codes:**
| Code | Description |
|------|-------------|
| 1 | Collision with parked vehicle |
| 2 | Rear-end / sideswipe (same direction) |
| **3** | **Head-on collision** ← high energy |
| 4 | Junction collision |
| 5 | Entering vehicle |
| 6 | Stationary traffic collision |
| **7** | **Run-off road** ← high energy |

In [ ]:
mw = df[df['on_motorway']].copy()
mw_unl = mw[mw['on_unlimited']]
mw_lim = mw[mw['on_limited_mw']]

utyp_labels = {
    1: 'Parked vehicle (1)',
    2: 'Rear-end / sideswipe (2)',
    3: 'Head-on (3)',
    4: 'Junction (4)',
    5: 'Entering vehicle (5)',
    6: 'Stationary traffic (6)',
    7: 'Run-off road (7)',
}

pct_unl = (mw_unl['UTYP1'].value_counts() / len(mw_unl) * 100).reindex(range(1,8), fill_value=0)
pct_lim = (mw_lim['UTYP1'].value_counts() / len(mw_lim) * 100).reindex(range(1,8), fill_value=0)

# Chi-squared test
contingency = pd.DataFrame({'unlimited': mw_unl['UTYP1'].value_counts(),
                             'limited':   mw_lim['UTYP1'].value_counts()}).fillna(0)
chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency.values)
print(f'Chi-squared test: χ²={chi2:.2f}, df={dof}, p={p_chi2:.2e}')

# High-energy proportions
he_unl = pct_unl[[3, 7]].sum()
he_lim = pct_lim[[3, 7]].sum()
print(f'\nHigh-energy proportions: Unlimited={he_unl:.1f}%, Limited={he_lim:.1f}%, Δ={he_unl-he_lim:+.1f}pp')

# Distribution table
df_dist = pd.DataFrame({'Unlimited (%)': pct_unl, 'Limited (%)': pct_lim}).rename(index=utyp_labels)
df_dist['Δ (pp)'] = df_dist['Unlimited (%)'] - df_dist['Limited (%)']
df_dist

In [ ]:
# Case fatality rate by UTYP1 × group
mw_c = mw[mw['on_unlimited'] | mw['on_limited_mw']].copy()
mw_c['is_fatal'] = (mw_c['severity'] == 'fatal').astype(int)
mw_c['group']    = mw_c['on_unlimited'].map({True: 'Unlimited', False: 'Limited'})

cfr = (
    mw_c.groupby(['UTYP1', 'group'])['is_fatal']
    .agg(cfr_pct=lambda x: x.mean() * 100, n='count')
    .reset_index()
)
cfr['label'] = cfr['UTYP1'].map(utyp_labels)
cfr_pivot = cfr.pivot(index='label', columns='group', values='cfr_pct')
print('Case fatality rate (%) by crash type × speed regime:')
cfr_pivot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: stacked bar
ax1 = axes[0]
x_pos = np.array([0, 1])
bottom = np.zeros(2)
type_colors = {1: '#adb5bd', 2: '#74b9ff', 3: '#d62728', 4: '#adb5bd',
               5: '#adb5bd', 6: '#636e72', 7: '#ff7675'}

for utyp in range(1, 8):
    heights = np.array([pct_unl[utyp], pct_lim[utyp]])
    ax1.bar(x_pos, heights, 0.5, bottom=bottom, color=type_colors[utyp],
            label=utyp_labels[utyp], edgecolor='white', linewidth=0.4)
    for i, (h, b) in enumerate(zip(heights, bottom)):
        if h > 3:
            ax1.text(x_pos[i], b + h/2, f'{h:.1f}%', ha='center', va='center',
                     fontsize=8, color='white' if utyp in (3, 7) else 'black',
                     fontweight='bold' if utyp in (3, 7) else 'normal')
    bottom += heights

ax1.set_xticks(x_pos)
ax1.set_xticklabels(['Unlimited Autobahn', 'Speed-limited Autobahn'], fontsize=11)
ax1.set_ylabel('Share of motorway accidents (%)')
ax1.set_title(f'Crash type distribution\nHigh-energy: Unlimited {he_unl:.1f}%, Limited {he_lim:.1f}%')
ax1.legend(loc='upper right', fontsize=7)
ax1.annotate('Doecke (2018): head-on safe ≤50 km/h | run-off safe ≤70 km/h',
             xy=(0.5, 0.02), xycoords='axes fraction', ha='center', fontsize=8, style='italic')

# Panel 2: CFR by crash type
ax2 = axes[1]
cfr_unl_vals = [cfr[(cfr['UTYP1']==u) & (cfr['group']=='Unlimited')]['cfr_pct'].values[0]
                if len(cfr[(cfr['UTYP1']==u) & (cfr['group']=='Unlimited')]) else 0
                for u in range(1, 8)]
cfr_lim_vals = [cfr[(cfr['UTYP1']==u) & (cfr['group']=='Limited')]['cfr_pct'].values[0]
                if len(cfr[(cfr['UTYP1']==u) & (cfr['group']=='Limited')]) else 0
                for u in range(1, 8)]

x2, w2 = np.arange(7), 0.35
ax2.bar(x2 - w2/2, cfr_unl_vals, w2, color='#e63946', label='Unlimited', alpha=0.85)
ax2.bar(x2 + w2/2, cfr_lim_vals, w2, color='#457b9d', label='Speed-limited', alpha=0.85)
ax2.set_xticks(x2)
ax2.set_xticklabels([utyp_labels[u].split(' (')[0].replace('/', '/\n') for u in range(1,8)], fontsize=8)
ax2.set_ylabel('Case fatality rate (%)')
ax2.set_title('Case fatality rate by collision type')
ax2.legend(fontsize=9)
for idx in [2, 6]:  # UTYP1 3 and 7 are indices 2 and 6
    ax2.axvspan(idx - 0.5, idx + 0.5, alpha=0.08, color='red', zorder=0)

fig.suptitle('H5: Safe System crash type analysis (Doecke 2018)', fontsize=13, fontweight='bold')
sns.despine(); plt.tight_layout()
plt.show()

### H5 Interpretation

| Finding | Value |
|---------|-------|
| High-energy crashes (head-on + run-off), unlimited | 14.9% |
| High-energy crashes, speed-limited | 14.1% |
| Difference | +0.8 percentage points |
| Chi-squared test (UTYP1 distribution) | χ² = 398, df = 6, p < 1e-80 |

**Key findings:**
- The overall UTYP1 distribution is **highly significantly different** between unlimited and limited sections (p < 10⁻⁸⁰), but the absolute high-energy share difference is small (+0.8pp).
- High-energy crashes on **unlimited sections have higher CFRs**: head-on 0.65% vs 0.32%, run-off 2.38% vs 1.52%.
- Junction accidents (UTYP1=4) show the most dramatic CFR difference: 26.2% on unlimited vs 12.8% on limited — but these are rare events on motorways.

**Conclusion:** H5 is **partially supported**:
1. The crash type *distribution* differs significantly, but the high-energy proportion gap is modest (+0.8pp).
2. More telling: high-energy crash types are **more lethal per incident** on unlimited sections — consistent with Doecke (2018) that crash outcomes degrade sharply above Safe System thresholds.
3. The combination of slightly more high-energy crashes + higher CFR per crash type both contribute to the overall elevated fatality rate on unlimited sections.

---
## Summary

| Hypothesis | Result | Key Finding |
|-----------|--------|-------------|
| **H3** Diverging trends | **Directionally supported, not statistically confirmed** | Limited sections improve faster (−4.6%/yr vs −2.7%/yr) but interaction p=0.24 |
| **H4** Power Model | **Not confirmed** | Observed ratios are below Power Model predictions, suggesting confounders (lower traffic, better geometry) on unlimited sections partially offset the speed effect |
| **H5** Crash types | **Partially supported** | Small proportion difference (+0.8pp high-energy), but consistently higher case fatality rates within each crash type on unlimited sections |

### Overall interpretation

The evidence collectively points to a **real but complex** relationship between unlimited speeds and fatality risk:

1. **Speed matters** — both the Johansson trend analysis and the Doecke CFR analysis show worse outcomes on unlimited sections.
2. **Confounders are important** — the Power Model systematically over-predicts the gap, suggesting unlimited sections have structural advantages (no junctions, lower density, better road design) that absorb some of the speed-related excess risk.
3. **Crash type alone doesn't explain the gap** — the 0.8pp difference in high-energy types is too small to account for the ~50% higher fatality rate index documented in prior notebooks.

A definitive causal analysis would require:
- Traffic volume data (vehicle-km travelled) for proper exposure normalisation
- Road geometry controls (curvature, gradient, lane width)
- A quasi-experimental design (e.g. sections that switched speed regime)

### References
- Johansson, G. (1996). *Speed and road fatalities: A study on 15 countries.*
- Elvik, R. (2013). A re-parameterisation of the Power Model of the relationship between the speed of traffic and the number of accidents and accident victims. *Accident Analysis & Prevention*, 50.
- Doecke, S., Kloeden, C., Dutschke, J., & Baldock, M. (2018). *Safe system speed limits for Australian roads.* CASR.
- BASt (2019). *Verkehr auf Bundesautobahnen 2019.* Bundesanstalt für Straßenwesen.